In [10]:
# Install necessary packages
# !pip install tiktoken
# !pip install transformers
!pip install datasets transformers
import re
from transformers import pipeline
from datasets import Dataset

# Load data
file_path = "/home/vpathaka/ITS_520/Training_VetGPT/combined_vetgpt_output.txt"
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

# Step 1: Basic Cleanup
def basic_cleanup(text):
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Remove extra spaces and newline characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text
clean_text = basic_cleanup(text)


Defaulting to user installation because normal site-packages is not writeable


In [11]:
# Step 2: Split text into manageable chunks for processing
def split_text(text, chunk_size=500):  # Reduce chunk size for coherence
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [12]:
# Prepare dataset with chunks
chunks = split_text(clean_text)
dataset = Dataset.from_dict({"text": chunks})

In [13]:
# Initialize the language model pipeline
corrector = pipeline("text2text-generation", model="t5-small", device=0)

# Function to apply the corrector to each batch
def correct_text(batch):
    results = corrector(batch["text"], max_length=1024, truncation=True)
    batch["corrected_text"] = [result["generated_text"] for result in results]
    return batch


In [ ]:
# Apply correction in batches using the dataset
corrected_dataset = dataset.map(correct_text, batched=True, batch_size=5)

# Combine all corrected chunks with spaces between them
final_text = "\n".join(corrected_dataset["corrected_text"])

# Post-process to add spaces if needed
final_text = re.sub(r'([a-z])([A-Z])', r'\1 \2', final_text)  # Add space before capital letters if needed

# Write the corrected output to a new file
with open("corrected_vet_data1.txt", 'w', encoding='utf-8') as f:
    f.write(final_text)

print("Data cleaned and saved to 'corrected_vet_data1.txt'")

Map:   0%|          | 0/3317 [00:00<?, ? examples/s]